# Домашнее задание 5

**Дисциплина** Алгоритмы и структуры данных

**Тема** Геометрические алгоритмы

# Задание 1. Ближайший объект

| Ограничение | Значение |
| ----------- | -------- |
| Ограничение времени | 1 секунда |
| Ограничение памяти | 64 Мб |
| Ввод | input.txt |
| Вывод | output.txt |

Вы находитесь в точке $(x_0, y_0)$. Вокруг вас находятся объекты, границы которых заданы многоугольниками или окружностями. У каждого объекта есть порядковый номер, соответствующий номеру в последовательности ввода. Требуется определить ближайший к вам объект. Если таких несколько, выберите тот, у которого номер меньше.

Гарантируется, что точка не лежит ни в одной из фигур.

**Формат ввода**

В первой строке через пробел указаны координаты вашей локации: $x_0$ и $y_0$, где $x_0, y_0 \in \mathbb{R}$.

Во второй строке записано целое число $n$ - количество фигур на плоскости.
В последующих $n$ строках дано описание фигур:
- для окружности описание начинается со слова `Circle`, далее через пробел даны координаты центра окружности $x_c, y_c$ и радиус $r$;
- для многоугольника строка начинается со слова `Polygon`, потом указано количество вершин $m$ и $m$ пар координат точек, задающих вершины многоугольника.

Параметры фигур - вещественные числа.

**Формат вывода**

Требуется вывести порядковый номер ближайшей фигуры (нумерация ведётся с единицы).

In [1]:
data = '''0 0
2
Circle 2 0 1
Polygon 4 5 0 5 1 6 1 6 0
'''

with open('input.txt', 'w', encoding='utf-8') as f:
    f.write(data)

In [2]:
import math

def distance_to_circle(x0, y0, xc, yc, r):
    dist_to_center = math.sqrt((x0 - xc)**2 + (y0 - yc)**2)
    return abs(dist_to_center - r)

def distance_point_to_segment(px, py, x1, y1, x2, y2):
    dx = x2 - x1
    dy = y2 - y1

    if dx == 0 and dy == 0:
        return math.sqrt((px - x1)**2 + (py - y1)**2)

    t = max(0, min(1, ((px - x1) * dx + (py - y1) * dy) / (dx * dx + dy * dy)))

    proj_x = x1 + t * dx
    proj_y = y1 + t * dy

    return math.sqrt((px - proj_x)**2 + (py - proj_y)**2)

def distance_to_polygon(x0, y0, vertices):
    min_dist = float('inf')
    n = len(vertices)

    for i in range(n):
        x1, y1 = vertices[i]
        x2, y2 = vertices[(i + 1) % n]
        dist = distance_point_to_segment(x0, y0, x1, y1, x2, y2)
        min_dist = min(min_dist, dist)

    return min_dist

with open('input.txt', 'r', encoding='utf-8') as f:
    x0, y0 = map(float, f.readline().strip().split())
    n = int(f.readline().strip())

    min_distance = float('inf')
    closest_figure = 1

    for i in range(1, n + 1):
        line = f.readline().strip().split()

        if line[0] == 'Circle':
            xc, yc, r = map(float, line[1:4])
            dist = distance_to_circle(x0, y0, xc, yc, r)
        else:
            m = int(line[1])
            vertices = []
            for j in range(m):
                x = float(line[2 + j * 2])
                y = float(line[3 + j * 2])
                vertices.append((x, y))
            dist = distance_to_polygon(x0, y0, vertices)

        if dist < min_distance:
            min_distance = dist
            closest_figure = i

with open('output.txt', 'w', encoding='utf-8') as f:
    f.write(str(closest_figure))

# Задание 2. Контур

| Ограничение | Значение |
| ----------- | -------- |
| Ограничение времени | 1 секунда |
| Ограничение памяти | 64 Мб |
| Ввод | input.txt |
| Вывод | output.txt |

В матрице $10 \times 10$ дано черно-белое изображение, где белые пиксели - это 1, а черные - 0. Нужно определить, есть ли на изображении белый замкнутый контур.

Белый контур - это замкнутая линия из единиц, полностью окружающая некоторую область из нулей.

Необходимо учитывать, что путь из любой точки контура в любую другую точку, принадлежащую этому контуру, возможен только при движении вверх, вниз, вправо или влево. Иначе считается, что контур прерывается (то есть в контуре не может быть участков, которые соприкасаются только по диагонали).

**Формат ввода**

На 10 строках даны элементы матрицы.

В каждой строке через пробел записаны 10 элементов.

**Формат вывода**

Вывести `YES`, если есть хотя бы один замкнутый белый контур.

Вывести `NO`, если контура нет.

In [3]:
data = '''0 0 0 0 0 0 0 0 0 1
0 0 0 0 0 0 0 0 1 1
0 0 0 0 0 0 0 1 0 1
0 0 0 0 0 0 1 0 0 1
0 0 0 0 0 1 0 0 0 1
0 0 0 0 1 0 0 0 0 1
0 0 0 1 0 0 0 0 0 1
0 0 1 0 0 0 0 0 0 1
0 1 0 0 0 0 0 0 0 1
1 1 1 1 1 1 1 1 1 1
'''

with open('input.txt', 'w', encoding='utf-8') as f:
    f.write(data)

In [4]:
from collections import deque

def main():
    n = 10
    m = 10

    a = [[0] * (m + 2) for _ in range(n + 2)]

    with open('input.txt', 'r', encoding='utf-8') as f:
        for i in range(1, n + 1):
            line = f.readline()
            if not line:
                return
            row = list(map(int, line.strip().split()))
            for j in range(1, m + 1):
                a[i][j] = row[j - 1]

    vis = [[False] * (m + 2) for _ in range(n + 2)]

    dx = [1, -1, 0, 0, 1, 1, -1, -1]
    dy = [0, 0, 1, -1, 1, -1, 1, -1]

    q = deque()
    q.append((0, 0))
    vis[0][0] = True

    while q:
        x, y = q.popleft()
        for dir in range(8):
            nx = x + dx[dir]
            ny = y + dy[dir]
            if nx < 0 or nx > n + 1 or ny < 0 or ny > m + 1:
                continue
            if vis[nx][ny]:
                continue
            if a[nx][ny] != 0:
                continue
            vis[nx][ny] = True
            q.append((nx, ny))

    has_hole = False
    for i in range(1, n + 1):
        for j in range(1, m + 1):
            if a[i][j] == 0 and not vis[i][j]:
                has_hole = True
                break
        if has_hole:
            break

    with open('output.txt', 'w', encoding='utf-8') as f:
        f.write("YES\n" if has_hole else "NO\n")

if __name__ == "__main__":
    main()